# Aggregated News Comparative Study

BoolQ is a Q/A dataset, containing 12697 examples sourced from google queries.

Each example is triplet of (question, passage, answer)

**Authors:** Anh Nguyen, Berke Kara

#Pre-Process

In [45]:
!pip install gensim datasets


from collections import Counter
from datasets import load_dataset,concatenate_datasets,Dataset, DatasetDict
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import pipeline

In [ ]:
ds = load_dataset("fancyzhx/ag_news")
ds_combined = concatenate_datasets([ds["train"], ds["test"]])
df = pd.DataFrame(ds_combined)
w2v_model = api.load("word2vec-google-news-300")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
df_sampled, _ = train_test_split(
    df,
    train_size=30000,
    random_state=42,
    stratify=df["label"]
)

train_df, temp_df = train_test_split(
    df_sampled,
    test_size=0.3,
    random_state=42,
    stratify=df_sampled["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

In [ ]:
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Test examples: {len(X_test)}")

# Model 1: Classical Baseline

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

val_preds = model.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation Macro F1:", f1_score(y_val, val_preds, average="macro"))

test_preds = model.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print("Test Macro F1:", f1_score(y_test, test_preds, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, test_preds))

# Model 2: Pretrained Word2Vec with Classifier

In [ ]:
def text_to_vector(text, model, dim=300):
    """
    Convert a sentence/document to a fixed-size vector
    by averaging all word vectors (mean pooling).
    """
    tokens = simple_preprocess(text)

    valid_vectors = [model[token] for token in tokens if token in model]

    if not valid_vectors:
        return np.zeros(dim)

    return np.mean(valid_vectors, axis=0)


In [ ]:
X = np.array([text_to_vector(text, w2v_model) for text in X_train])
y = np.array(y_train)

clf = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500)

clf.fit(X, y)
print("Classifier trained")


In [ ]:
X_test_vec = np.array([text_to_vector(text, w2v_model) for text in X_test])
X_val_vec = np.array([text_to_vector(text, w2v_model) for text in X_val])

y_pred = clf.predict(X_test_vec)

print("Word2Vec + Classifier Baseline")
print("-" * 50)

val_preds = clf.predict(X_val_vec)
print(f"Validation Accuracy:, {accuracy_score(y_val, val_preds):0.4f}")
print(f"Validation Macro F1:, {f1_score(y_val, val_preds, average="macro"):0.4f}")

test_preds = clf.predict(X_test_vec)
print(f"Test Accuracy:, {accuracy_score(y_test, test_preds):0.4f}")
print(f"Test Macro F1:, {f1_score(y_test, test_preds, average="macro"):0.4f}")

print("\nClassification Report:")
print(classification_report(y_test, test_preds))


# Model 3: LSTM from Scratch

In [ ]:
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from gensim.utils import simple_preprocess

# 1. Build Vocabulary
def build_vocab(texts, max_size=20000):
    counter = Counter()
    for text in texts:
        counter.update(simple_preprocess(text))

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
print(f"Vocabulary size: {len(vocab)}")

# 2. Dataset Class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts.tolist()
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = simple_preprocess(self.texts[idx])
        indices = [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]
        if len(indices) == 0:
            indices = [self.vocab['<UNK>']] # Handle empty sequences
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# 3. Collate Function (for padding)
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    # Pad sequences to match the longest one in the batch
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, lengths, torch.stack(labels)

# 4. Create DataLoaders
train_dataset = TextDataset(X_train, y_train, vocab)
test_dataset = TextDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created test_loader with {len(test_loader)} batches.")

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded: (batch, seq_len, embed_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, _) = self.lstm(packed)

        # hidden: (num_layers * 2, batch, hidden_dim)
        forward_hidden  = hidden[-2]   # last layer, forward
        backward_hidden = hidden[-1]   # last layer, backward

        # Concatenate both directions
        combined = torch.cat([forward_hidden, backward_hidden], dim=1)
        # combined: (batch, hidden_dim * 2)

        out = self.dropout(combined)
        # (batch, num_classes)
        return self.fc(out)

model = LSTMClassifier(
    vocab_size  = len(vocab),
    embed_dim   = 128,
    hidden_dim  = 256,
    num_classes = 4,
    num_layers  = 2,
    dropout     = 0.3
).to(device)

# Count parameters — report this in your paper
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for sequences, lengths, labels in loader:
        sequences, lengths, labels = (
            sequences.to(device),
            lengths.to(device),
            labels.to(device)
        )

        optimizer.zero_grad()
        outputs = model(sequences, lengths)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping — important for LSTM stability
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for sequences, lengths, labels in loader:
            sequences, lengths, labels = (
                sequences.to(device),
                lengths.to(device),
                labels.to(device)
            )
            outputs = model(sequences, lengths)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

# Train
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = evaluate(model, test_loader, criterion)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for sequences, lengths, labels in test_loader:
        outputs = model(sequences.to(device), lengths.to(device))
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("LSTM from Scratch — Final Results")
print("-" * 50)
print(classification_report(all_labels, all_preds,
      target_names=["World", "Sports", "Business", "Sci/Tech"]))


# Model 4: Pretrained Model with Fine-Tuning

---



In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_agnews",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

predictions = trainer.predict(tokenized_dataset["validation"])

logits = predictions.predictions
labels = predictions.label_ids

# val results
preds = np.argmax(logits, axis=1)

accuracy = accuracy_score(labels, preds)
macro_f1 = f1_score(labels, preds, average="macro")

print("Validation Accuracy:", accuracy)
print("Validation Macro F1:", macro_f1)

# test results

test_preds = trainer.predict(tokenized_dataset["test"])

preds = np.argmax(test_preds.predictions, axis=1)
labels = test_preds.label_ids

print("Test Accuracy:", accuracy_score(labels, preds))
print("Test Macro F1:", f1_score(labels, preds, average="macro"))









# Model 5: pretrained model without task-specific training

In [ ]:

id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

label2id = {v: k for k, v in id2label.items()}
candidate_labels = list(label2id.keys())

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

def predict_zero_shot(texts, batch_size=16):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        outputs = zero_shot_classifier(
            batch,
            candidate_labels=candidate_labels,
            hypothesis_template="This news article is about {}.",
            multi_label=False
        )


        if isinstance(outputs, dict):
            outputs = [outputs]

        for output in outputs:
            pred_label = output["labels"][0]
            preds.append(label2id[pred_label])

        print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} examples")

    return np.array(preds)

In [ ]:
val_texts = X_val.tolist()
val_labels = y_val.to_numpy()

zero_val_preds = predict_zero_shot(val_texts, batch_size=32)

val_accuracy = accuracy_score(val_labels, zero_val_preds)
val_macro_f1 = f1_score(val_labels, zero_val_preds, average="macro")
val_weighted_f1 = f1_score(val_labels, zero_val_preds, average="weighted")

print("Zero-Shot Validation Accuracy:", val_accuracy)
print("Zero-Shot Validation Macro F1:", val_macro_f1)
print("Zero-Shot Validation Weighted F1:", val_weighted_f1)

print("\nValidation Classification Report:")
print(classification_report(
    val_labels,
    zero_val_preds,
    target_names=[id2label[i] for i in range(4)]
))

In [ ]:
test_texts = X_test.tolist()
test_labels = y_test.to_numpy()

zero_test_preds = predict_zero_shot(test_texts, batch_size=16)

test_accuracy = accuracy_score(test_labels, zero_test_preds)
test_macro_f1 = f1_score(test_labels, zero_test_preds, average="macro")
test_weighted_f1 = f1_score(test_labels, zero_test_preds, average="weighted")

print("Zero-Shot Test Accuracy:", test_accuracy)
print("Zero-Shot Test Macro F1:", test_macro_f1)
print("Zero-Shot Test Weighted F1:", test_weighted_f1)

print("\nTest Classification Report:")
print(classification_report(
    test_labels,
    zero_test_preds,
    target_names=[id2label[i] for i in range(4)]
))

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, zero_test_preds))